# 🏋️ Data Collection & Web Scraping — Practical Exercises

> **Hands-on Practice**: REST API consumption, HTML table parsing, attribute extraction, rate-limited crawling aur product card scraping.


## ❓ Problem 1: Public REST API Consumption & JSON Normalization
- **Problem:** Public Quotes API consume karke `quote` aur `author` columns ka DataFrame banao.
- **Concept:** `requests.get()` $\to$ `.json()` $\to$ `pd.DataFrame()`.


In [1]:
import requests
import pandas as pd

api_url = "https://dummyjson.com/quotes?limit=5"

try:
    res = requests.get(api_url, timeout=5)
    res.raise_for_status()
    quotes_data = res.json()["quotes"]
    print("Fetched 5 Quotes Online!")
except Exception:
    # Offline Fallback Simulation
    quotes_data = [
        {"id": 1, "quote": "Life is what happens when you're busy making other plans.", "author": "John Lennon"},
        {"id": 2, "quote": "The way to get started is to quit talking and begin doing.", "author": "Walt Disney"},
        {"id": 3, "quote": "Your time is limited, so don't waste it living someone else's life.", "author": "Steve Jobs"},
        {"id": 4, "quote": "If life were predictable it would cease to be life.", "author": "Eleanor Roosevelt"},
        {"id": 5, "quote": "Spread love everywhere you go.", "author": "Mother Teresa"}
    ]
    print("Loaded 5 Quotes from Offline Simulation.")

df_quotes = pd.DataFrame(quotes_data)[["id", "quote", "author"]]
print("\n--- Clean Quotes Table ---")
display(df_quotes)


Fetched 5 Quotes Online!

--- Clean Quotes Table ---


,id,quote,author
0,1,Your heart is the size of an ocean. Go find yo...,Rumi
1,2,The Bay of Bengal is hit frequently by cyclone...,Abdul Kalam
2,3,"Thinking is the capital, Enterprise is the way...",Abdul Kalam
3,4,"If You Can'T Make It Good, At Least Make It Lo...",Bill Gates
4,5,"Heart be brave. If you cannot be brave, just g...",Rumi


## ❓ Problem 2: HTML Tabular Data Extraction into DataFrame
- **Problem:** Raw HTML table me se rows (`tr`) aur cells (`td`) parse karke stock market table banao.
- **Concept:** `soup.find('table')` aur list comprehension se DataFrame construction.


In [2]:
from bs4 import BeautifulSoup

html_table_snippet = """
<table class="market-data">
    <thead>
        <tr><th>Ticker</th><th>Price</th><th>Change</th><th>Volume</th></tr>
    </thead>
    <tbody>
        <tr><td>NVDA</td><td>$135.50</td><td>+4.2%</td><td>52M</td></tr>
        <tr><td>AAPL</td><td>$228.10</td><td>-0.8%</td><td>38M</td></tr>
        <tr><td>GOOGL</td><td>$178.40</td><td>+1.5%</td><td>24M</td></tr>
        <tr><td>MSFT</td><td>$445.00</td><td>+0.3%</td><td>19M</td></tr>
    </tbody>
</table>
"""

soup = BeautifulSoup(html_table_snippet, "html.parser")
table = soup.find("table", class_="market-data")

# Extract headers
headers = [th.get_text(strip=True) for th in table.find("thead").find_all("th")]

# Extract rows
rows = []
for tr in table.find("tbody").find_all("tr"):
    cells = [td.get_text(strip=True) for td in tr.find_all("td")]
    rows.append(cells)

df_stocks = pd.DataFrame(rows, columns=headers)
print("--- Parsed Market Data Table ---")
display(df_stocks)


--- Parsed Market Data Table ---


,Ticker,Price,Change,Volume
0,NVDA,$135.50,+4.2%,52M
1,AAPL,$228.10,-0.8%,38M
2,GOOGL,$178.40,+1.5%,24M
3,MSFT,$445.00,+0.3%,19M


## ❓ Problem 3: Extracting Hyperlinks & Image URLs (Attributes)
- **Problem:** Webpage me se image banner source (`src`) aur related blog links (`href`) extract karo.
- **Concept:** Tag attribute dictionary lookup `tag['href']` aur `tag['src']`.


In [3]:
nav_html = """
<div class="blog-post">
    <h2>Top AI Breakthroughs</h2>
    <img src="/static/images/banner.jpg" alt="AI Banner">
    <div class="references">
        <a href="https://arxiv.org/abs/transformer" class="ref-link">Attention Is All You Need</a>
        <a href="https://openai.com/research" class="ref-link">OpenAI Research</a>
        <a href="https://deepmind.google/discover" class="ref-link">DeepMind Blog</a>
    </div>
</div>
"""

blog_soup = BeautifulSoup(nav_html, "html.parser")

# 1. Image Source
img_src = blog_soup.find("img")["src"]
print("Banner Image URL:", img_src)

# 2. All Hyperlinks
links = []
for a in blog_soup.find_all("a", class_="ref-link"):
    links.append({
        "Title": a.get_text(strip=True),
        "URL": a["href"]
    })

df_links = pd.DataFrame(links)
print("\n--- Extracted References ---")
display(df_links)


Banner Image URL: /static/images/banner.jpg

--- Extracted References ---


,Title,URL
0,Attention Is All You Need,https://arxiv.org/abs/transformer
1,OpenAI Research,https://openai.com/research
2,DeepMind Blog,https://deepmind.google/discover


## ❓ Problem 4: Rate-Limited Safe Scraper with `time.sleep`
- **Problem:** 3 pages crawl karte waqt server overload prevent karne ke liye rate limiting introduce karo.
- **Concept:** `time.sleep(1.0)` delay loop ethics.


In [4]:
import time

pages = [1, 2, 3]
scraped_logs = []

print("Starting Rate-Limited Crawler:")
for page in pages:
    start_ts = time.time()
    
    # 0.2 second delay simulation
    time.sleep(0.2)
    
    scraped_logs.append({
        "Page": page,
        "Status": 200,
        "Items_Extracted": page * 10,
        "Latency_Sec": round(time.time() - start_ts, 3)
    })
    print(f"  [OK] Page {page} scraped successfully with safety delay.")

df_crawler = pd.DataFrame(scraped_logs)
print("\n--- Crawler Execution Logs ---")
display(df_crawler)


Starting Rate-Limited Crawler:


  [OK] Page 1 scraped successfully with safety delay.


  [OK] Page 2 scraped successfully with safety delay.


  [OK] Page 3 scraped successfully with safety delay.

--- Crawler Execution Logs ---


,Page,Status,Items_Extracted,Latency_Sec
0,1,200,10,0.211
1,2,200,20,0.201
2,3,200,30,0.200


## ❓ Problem 5: E-Commerce Product Cards Scraper to Clean CSV
- **Problem:** HTML product grid me se title, rating, aur discount price nikal kar CSV file me export karo.
- **Concept:** Nested parsing loop $\to$ Data Cleaning $\to$ `.to_csv()`.


In [5]:
catalog_html = """
<div class="products-grid">
    <div class="product-item">
        <h4 class="title">Wireless Noise-Cancelling Headphones</h4>
        <div class="rating">4.8 Stars (1,240 Reviews)</div>
        <div class="price-box">
            <span class="original-price">$299.99</span>
            <span class="sale-price">$199.99</span>
        </div>
    </div>
    <div class="product-item">
        <h4 class="title">Ergonomic Mechanical Keyboard</h4>
        <div class="rating">4.6 Stars (850 Reviews)</div>
        <div class="price-box">
            <span class="original-price">$149.99</span>
            <span class="sale-price">$119.50</span>
        </div>
    </div>
    <div class="product-item">
        <h4 class="title">Ultra-Wide 4K Monitor</h4>
        <div class="rating">4.9 Stars (3,100 Reviews)</div>
        <div class="price-box">
            <span class="original-price">$699.99</span>
            <span class="sale-price">$549.00</span>
        </div>
    </div>
</div>
"""

cat_soup = BeautifulSoup(catalog_html, "html.parser")
parsed_items = []

for card in cat_soup.find_all("div", class_="product-item"):
    title = card.find("h4", class_="title").get_text(strip=True)
    rating_text = card.find("div", class_="rating").get_text(strip=True)
    orig_price = float(card.find("span", class_="original-price").get_text(strip=True).replace("$", ""))
    sale_price = float(card.find("span", class_="sale-price").get_text(strip=True).replace("$", ""))
    discount_pct = round(((orig_price - sale_price) / orig_price) * 100, 1)
    
    parsed_items.append({
        "Product": title,
        "Rating": rating_text.split(" ")[0],
        "Original_Price": orig_price,
        "Sale_Price": sale_price,
        "Discount_Percent": f"{discount_pct}%"
    })

df_products = pd.DataFrame(parsed_items)
print("--- Extracted E-Commerce Products ---")
display(df_products)

# Export to clean CSV
out_csv = "scraped_electronics.csv"
df_products.to_csv(out_csv, index=False)

import os
print(f"\nSuccessfully exported to '{out_csv}' | Exists: {os.path.exists(out_csv)}")


--- Extracted E-Commerce Products ---


,Product,Rating,Original_Price,Sale_Price,Discount_Percent
0,Wireless Noise-Cancelling Headphones,4.8,299.99,199.99,33.3%
1,Ergonomic Mechanical Keyboard,4.6,149.99,119.50,20.3%
2,Ultra-Wide 4K Monitor,4.9,699.99,549.00,21.6%



Successfully exported to 'scraped_electronics.csv' | Exists: True


---

## 🎯 Practice Exercises Summary Checklist
- [x] **Problem 1:** Public API consumption & tabular normalization.
- [x] **Problem 2:** HTML table parsing (`tr`/`td`) into DataFrame.
- [x] **Problem 3:** Attribute scraping (`href`, `src`).
- [x] **Problem 4:** Rate-limiting ethics with `time.sleep`.
- [x] **Problem 5:** Product cards parsing with discount calculation & CSV export.

---
### 🏆 Web Scraping Practical Exercises Completed!
